In [1]:
import os
os.chdir('../../../../..')

In [2]:
from src.datasets import MaterialsProject

In [3]:
import numpy as np

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN, KMeans
from kmedoids import KMedoids

from src.helper_functions import create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster

In [4]:
mp = MaterialsProject(limit=5000, sampling_strategy="stratified", stratify_on=["band_gap", "energy_above_hull"], add_mace=True)
df = mp.load()

2026-04-21 09:33:29.945 | INFO     | src.datasets:load:1317 - Loading full cached Parquet data from data/Materials Project/materials.parquet...
2026-04-21 09:33:30.849 | INFO     | src.datasets:_add_descriptors:1666 - Ignoring output_tag=sample_n6000_seed40_stratified since descriptors are attached directly to dataframe.
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))
2026-04-21 09:33:31.012 | INFO     | src.datasets:_add_descriptors:1706 - Loading MACE-MP model...
/Users/karlfindhansen/thesis/Anomaly-Detection-in-Molecular-and-Materials-Datasets/.venv/lib/python3.12/site-packages/mace/calculators/mace.py:199: UserWar

cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
Using Materials Project MACE for MACECalculator with /Users/karlfindhansen/.cache/mace/20231203mace128L1_epoch199model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.


2026-04-21 09:36:05.478 | INFO     | src.datasets:_add_descriptors:1753 - Computing MACE chunk 1 (1000 to 2000)...
2026-04-21 09:38:25.815 | INFO     | src.datasets:_add_descriptors:1753 - Computing MACE chunk 2 (2000 to 3000)...
2026-04-21 09:40:32.709 | INFO     | src.datasets:_add_descriptors:1753 - Computing MACE chunk 3 (3000 to 4000)...
2026-04-21 09:42:51.348 | INFO     | src.datasets:_add_descriptors:1753 - Computing MACE chunk 4 (4000 to 5000)...
2026-04-21 09:47:44.916 | INFO     | src.datasets:_add_descriptors:1753 - Computing MACE chunk 5 (5000 to 6000)...
2026-04-21 09:52:59.442 | SUCCESS  | src.datasets:_add_descriptors:1772 - All requested descriptors successfully added to dataframe.
2026-04-21 09:52:59.470 | SUCCESS  | src.datasets:load:1357 - Successfully reached requested limit of 5000 valid rows (Attempt 1).


In [9]:
dist_matrix = mp.get_distance_matrix(
    descriptor="mace",
    dist_type="euclidean",
    force_calculate=True,
    pca_variance=60,
)

2026-04-21 09:54:35.059 | INFO     | src.datasets:get_distance_matrix:1929 - Applying PCA to retain 60.00% of variance.
2026-04-21 09:54:35.497 | INFO     | src.datasets:get_distance_matrix:1938 - PCA reduced 'mace' dimensions from 256 to 9
2026-04-21 09:54:35.602 | INFO     | src.datasets:get_distance_matrix:1949 - Calculating distance matrix for mace using euclidean distance.
2026-04-21 09:54:36.060 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/Materials Project/dist_mace_euclidean_pca0.6.npy


# Hierarchical Clustering on Distance Matrix

In [10]:
model_hier = AgglomerativeClustering(metric='precomputed', n_clusters=3, linkage='complete')
labels_hier = model_hier.fit_predict(dist_matrix)
print(np.unique(labels_hier, return_counts=True))
df = df.with_columns(labels_hier=labels_hier)

(array([0, 1, 2]), array([3930,  266,  804]))


In [14]:
d = average_numeric_by_cluster(df, "labels_hier")

shape: (3, 19)
┌─────────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬────────┬────────┬────────┬─────────┬─────────┬─────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬───────────┐
│ labels_hier ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a      ┆ b      ┆ c      ┆ alpha   ┆ beta    ┆ gamma   ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ pct_metal │
│ ---         ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---       │
│ i64         ┆ u32   ┆ f64         ┆ f64             ┆ f64                       ┆ f64      ┆ f64     ┆ f64    ┆ f64    ┆ f64    ┆ f64     ┆ f64     ┆ f64     ┆ f64      ┆ f64       ┆ f64               ┆ f64             ┆ f64

In [12]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="mace",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_hier,
    clustering_method="hierarchical"
)

2026-04-21 09:56:28.639 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/materials_project/clustering/euclidean/mace/pca_hierarchical_projection.png


{'coords': array([[ -4.42965486,  -8.66439245],
        [ 30.42157934,   2.27176655],
        [ 17.64280794,  15.19923499],
        ...,
        [-18.71784459,   4.51028284],
        [-17.93054394,   6.62245579],
        [-17.12699583,   3.63018153]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/materials_project/clustering/euclidean/mace/pca_hierarchical_projection.png'),
 'output_dir': PosixPath('figures/materials_project/clustering/euclidean/mace'),
 'clustering_method': 'hierarchical'}

In [13]:
create_chemiscope_viewer(df, dist_matrix, labels_hier, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

# KMedoids

In [15]:
model_km = KMedoids(n_clusters=3, metric="precomputed")
labels_km = model_km.fit_predict(dist_matrix)
print(np.unique(labels_km, return_counts=True))
df = df.with_columns(labels_km=labels_km)

(array([0, 1, 2], dtype=uint64), array([1295, 1582, 2123]))


In [19]:
d = average_numeric_by_cluster(df, "labels_km")

shape: (3, 20)
┌───────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬────────┬────────┬────────┬─────────┬─────────┬─────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬─────────────┬───────────┐
│ labels_km ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a      ┆ b      ┆ c      ┆ alpha   ┆ beta    ┆ gamma   ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ labels_hier ┆ pct_metal │
│ ---       ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---         ┆ ---       │
│ u64       ┆ u32   ┆ f64         ┆ f64             ┆ f64                       ┆ f64      ┆ f64     ┆ f64    ┆ f64    ┆ f64    ┆ f64     ┆ f64     ┆ f64     ┆ f64      ┆ f64       ┆ f64    

In [17]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="mace",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_km,
    clustering_method="kmedoids"
)

2026-04-21 09:59:05.121 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/materials_project/clustering/euclidean/mace/pca_kmedoids_projection.png


{'coords': array([[ -4.42965486,  -8.66439245],
        [ 30.42157934,   2.27176655],
        [ 17.64280794,  15.19923499],
        ...,
        [-18.71784459,   4.51028284],
        [-17.93054394,   6.62245579],
        [-17.12699583,   3.63018153]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/materials_project/clustering/euclidean/mace/pca_kmedoids_projection.png'),
 'output_dir': PosixPath('figures/materials_project/clustering/euclidean/mace'),
 'clustering_method': 'kmedoids'}

In [18]:
create_chemiscope_viewer(df, dist_matrix, labels_km, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

# Spectral

In [20]:
model_spectral = SpectralClustering(
                n_clusters=3,
                affinity="precomputed",
                assign_labels='kmeans',
                random_state=42,
            )

labels_spectral = model_spectral.fit_predict(dist_matrix)
df = df.with_columns(labels_spectral=labels_spectral)

In [21]:
create_chemiscope_viewer(df, dist_matrix, labels_spectral, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

In [22]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="mace",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_spectral,
    clustering_method="spectral"
)

2026-04-21 11:02:41.321 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/materials_project/clustering/euclidean/mace/pca_spectral_projection.png


{'coords': array([[ -4.42965486,  -8.66439245],
        [ 30.42157934,   2.27176655],
        [ 17.64280794,  15.19923499],
        ...,
        [-18.71784459,   4.51028284],
        [-17.93054394,   6.62245579],
        [-17.12699583,   3.63018153]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/materials_project/clustering/euclidean/mace/pca_spectral_projection.png'),
 'output_dir': PosixPath('figures/materials_project/clustering/euclidean/mace'),
 'clustering_method': 'spectral'}

In [17]:
average_numeric_by_cluster(df, "labels_spectral")

shape: (3, 20)
┌─────────────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬────────┬────────┬────────┬─────────┬─────────┬─────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬─────────────┬───────────┐
│ labels_spectral ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a      ┆ b      ┆ c      ┆ alpha   ┆ beta    ┆ gamma   ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ labels_hier ┆ labels_km │
│ ---             ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---         ┆ ---       │
│ i32             ┆ u32   ┆ f64         ┆ f64             ┆ f64                       ┆ f64      ┆ f64     ┆ f64    ┆ f64    ┆ f64    ┆ f64     ┆ f64     ┆ f64     ┆ f64   

# DBSCAN

In [47]:
model_db = DBSCAN(
    eps=0.9,
    min_samples=2,
    metric='precomputed',
)

labels_db = model_db.fit_predict(dist_matrix)
df = df.with_columns(labels_db=labels_db)
print(np.unique(labels_db,return_counts=True))

(array([-1,  0,  1,  2,  3,  4,  5]), array([  40, 4940,    5,    3,    2,    7,    3]))


In [48]:
create_chemiscope_viewer(df, dist_matrix, labels_db, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

In [49]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="mace",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="materials_project",
    labels=labels_db,
    clustering_method="dbscan"
)

2026-04-19 16:26:36.426 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/materials_project/clustering/euclidean/mace/pca_dbscan_projection.png


{'coords': array([[  0.92011016,  -1.85708368],
        [ 24.15351562,  -7.48385943],
        [ 16.55228989,   4.20523489],
        ...,
        [-18.91730874,   4.87198451],
        [-20.31475448,   6.67796428],
        [-15.68066631,   4.66707103]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/materials_project/clustering/euclidean/mace/pca_dbscan_projection.png'),
 'output_dir': PosixPath('figures/materials_project/clustering/euclidean/mace'),
 'clustering_method': 'dbscan'}

In [21]:
average_numeric_by_cluster(df, "labels_db")

shape: (1, 21)
┌───────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬────────┬────────┬────────┬─────────┬─────────┬─────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬─────────────┬───────────┬─────────────────┐
│ labels_db ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a      ┆ b      ┆ c      ┆ alpha   ┆ beta    ┆ gamma   ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ labels_hier ┆ labels_km ┆ labels_spectral │
│ ---       ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---         ┆ ---       ┆ ---             │
│ i64       ┆ u32   ┆ f64         ┆ f64             ┆ f64                       ┆ f64      ┆ f64     ┆ f64    ┆ f64    ┆ f64    ┆ f64   

# KMeans on Raw Embeddings

In [22]:
X_raw = np.array(df["mace_embedding"].to_list(), dtype=np.float32)
kmeans_raw = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_kmeans_raw = kmeans_raw.fit_predict(X_raw)
df = df.with_columns(labels_kmeans_raw=labels_kmeans_raw)

In [26]:
create_chemiscope_viewer(df, X_raw, labels_kmeans_raw, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: materials_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show materials_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'Materials Project - PCA Clustering'}, settings={'map': {'x': {'property': 'PC…

In [24]:
average_numeric_by_cluster(df, "labels_kmeans_raw")

shape: (2, 22)
┌───────────────────┬───────┬─────────────┬─────────────────┬───────────────────────────┬──────────┬─────────┬────────┬────────┬────────┬─────────┬─────────┬─────────┬──────────┬───────────┬───────────────────┬─────────────────┬─────────────────┬─────────────┬───────────┬─────────────────┬───────────┐
│ labels_kmeans_raw ┆ count ┆ max_en_diff ┆ energy_per_atom ┆ formation_energy_per_atom ┆ band_gap ┆ density ┆ a      ┆ b      ┆ c      ┆ alpha   ┆ beta    ┆ gamma   ┆ volume   ┆ num_sites ┆ energy_above_hull ┆ avg_bond_length ┆ max_bond_length ┆ labels_hier ┆ labels_km ┆ labels_spectral ┆ labels_db │
│ ---               ┆ ---   ┆ ---         ┆ ---             ┆ ---                       ┆ ---      ┆ ---     ┆ ---    ┆ ---    ┆ ---    ┆ ---     ┆ ---     ┆ ---     ┆ ---      ┆ ---       ┆ ---               ┆ ---             ┆ ---             ┆ ---         ┆ ---       ┆ ---             ┆ ---       │
│ i32               ┆ u32   ┆ f64         ┆ f64             ┆ f64           